# Pipeline Batch: Bronze, Silver, Quarantine y Gold

Este notebook ejecuta las reglas de limpieza y agregación de `src/` sobre las 7 fuentes CSV y los 120 archivos JSONL (`43.200` eventos), generando los marts en `data/gold/`, los rechazos en `data/quarantine/` y el resumen en `evidence/batch_pipeline_resumen.json`.

Para correr la versión en PySpark sobre Google Colab o clúster Spark se usa `src/pyspark_etl.py`.

In [1]:
import json
import sys
from pathlib import Path

repo_root = Path.cwd()
if not (repo_root / 'config' / 'pipeline_config.json').exists() and (repo_root.parent / 'config' / 'pipeline_config.json').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.config_loader import load_pipeline_config
from src.schemas import SCHEMA_CONTRACTS

config = load_pipeline_config()
print('Esquemas definidos:', list(SCHEMA_CONTRACTS.keys()))
print('Particionamiento para series de tiempo:', config['partitioning']['time_series_datasets']['partition_by'])

Esquemas definidos: ['usage_events', 'customers_orgs', 'billing_monthly', 'resources', 'users', 'support_tickets', 'marketing_touches', 'nps_surveys']
Particionamiento para series de tiempo: ['year', 'month']


## 1. Corrida del pipeline y conteos por capa (Bronze, Silver y Quarantine)

In [2]:
from src.batch_pipeline import run_batch_pipeline

report = run_batch_pipeline()
print(json.dumps({
    'bronze_counts': report['bronze_counts'],
    'silver_and_quarantine_counts': report['silver_and_quarantine_counts'],
    'idempotency_verification': report['idempotency_verification'],
}, indent=2, ensure_ascii=False))

{
  "bronze_counts": {
    "usage_events_jsonl_files": 120,
    "usage_events_raw_rows": 43200,
    "usage_events_corrupt_json_rows": 0,
    "customers_orgs_raw_rows": 80,
    "billing_monthly_raw_rows": 240,
    "resources_raw_rows": 400,
    "users_raw_rows": 800,
    "support_tickets_raw_rows": 1000,
    "marketing_touches_raw_rows": 1500,
    "nps_surveys_raw_rows": 92
  },
  "silver_and_quarantine_counts": {
    "usage_events_silver_valid": 42984,
    "usage_events_quarantined_negative_cost": 216,
    "usage_events_unit_imputed_count": 2069,
    "usage_events_value_null_preserved_count": 873,
    "customers_orgs_silver": 80,
    "customers_orgs_invalid_nps_nullified": 1,
    "resources_silver": 400,
    "billing_monthly_silver": 240,
    "billing_monthly_credits_imputed": 137,
    "billing_monthly_credit_notes_preserved": 13,
    "support_tickets_silver": 1000,
    "support_tickets_csat_nullified_total": 203,
    "users_silver": 800,
    "users_inconsistent_login_nullified": 232,


## 2. Detección de spikes de costo: MAD (> 3,5) vs. Z-Score (> 3,0) por `(service, metric)`

In [3]:
print(json.dumps(report['anomaly_detection_comparison'], indent=2, ensure_ascii=False))

{
  "grouping_grain": [
    "service",
    "metric"
  ],
  "mad_modified_zscore_threshold": 3.5,
  "classic_zscore_threshold": 3.0,
  "mad_spikes_detected": 191,
  "classic_zscore_spikes_detected": 153,
  "spikes_detected_by_both": 152,
  "stats_by_service_metric": {
    "analytics__cpu_hours": {
      "count": 1146.0,
      "median": 0.12825,
      "mad": 0.071,
      "mean": 0.142506,
      "std": 0.189266
    },
    "analytics__requests": {
      "count": 2063.0,
      "median": 8.3594,
      "mad": 0.9562,
      "mean": 9.086811,
      "std": 11.270275
    },
    "analytics__storage_gb_hours": {
      "count": 1362.0,
      "median": 0.70695,
      "mad": 0.3824,
      "mean": 0.776272,
      "std": 1.410134
    },
    "compute__cpu_hours": {
      "count": 3125.0,
      "median": 0.1406,
      "mad": 0.083,
      "mean": 0.160953,
      "std": 0.261956
    },
    "compute__requests": {
      "count": 5545.0,
      "median": 9.5306,
      "mad": 1.1721,
      "mean": 10.484757,
   

## 3. Resumen de los marts generados en `data/gold/`

In [4]:
print(json.dumps(report['gold_marts_summary'], indent=2, ensure_ascii=False))

{
  "finops_org_service_daily_rows": 11050,
  "finops_org_service_monthly_rows": 524,
  "support_org_sla_severity_rows": 709,
  "product_genai_adoption_daily_rows": 4580,
  "cost_anomalies_by_org_rows": 191,
  "total_valid_cost_usd": 148351.6009,
  "total_carbon_kg": 365.5609,
  "total_genai_tokens": 2545481,
  "top_5_orgs_by_cost_usd": [
    {
      "org_id": "org_53lc58dr",
      "org_name": "Apex Digital 16",
      "plan": "standard",
      "events_count": 1109,
      "total_cost_usd": 4389.4683,
      "total_carbon_kg": 9.9666,
      "mad_spike_count": 1
    },
    {
      "org_id": "org_chj755nf",
      "org_name": "Delta Tech 32",
      "plan": "pro",
      "events_count": 955,
      "total_cost_usd": 4367.5798,
      "total_carbon_kg": 8.5235,
      "mad_spike_count": 5
    },
    {
      "org_id": "org_kdgigatj",
      "org_name": "Zenith Tech 79",
      "plan": "standard",
      "events_count": 1222,
      "total_cost_usd": 4351.4633,
      "total_carbon_kg": 10.603,
      "ma